# AAI614: Data Science & its Applications

*Notebook 2.5: Practice with Parquet and File Types*

<a href="https://colab.research.google.com/github/harmanani/AAI614/blob/main/Week%202/Notebook2.5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import time
import ssl

ssl._create_default_https_context = ssl._create_unverified_context

In [7]:
class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, *args):
        self.end = time.perf_counter()
        self.interval = self.end - self.start

##### Read the Parqeut file and time it

In [8]:
with Timer() as t_pd:
    df = pd.read_parquet('https://raw.githubusercontent.com/harmanani/AAI614/main/Week%202/niaaa-report.parquet')
print(t_pd.interval)

0.03940831600016281


In [19]:
df.head()
df.count()

,0
State,1683
Year,1683
Beer,1683
Wine,1683
Spirits,1683


#### Read the CSV file and time it

In [9]:
with Timer() as t_pd:
    df = pd.read_csv('https://raw.githubusercontent.com/harmanani/AAI614/main/Week%202/niaaa-report.csv')
print(t_pd.interval)

0.04389820799997324


In [15]:
df.head()
df.count()

,0
State,1683
Year,1683
Beer,1683
Wine,1683
Spirits,1683


#### Read the ZIP file and time it

In [10]:
import zipfile
with Timer() as t_pd:
    df = pd.read_csv('https://raw.githubusercontent.com/harmanani/AAI614/main/Week%202/niaaa-report.zip', compression="zip")
print(t_pd.interval)

0.036878826000020126


In [16]:
df.head()
df.count()

,0
State,1683
Year,1683
Beer,1683
Wine,1683
Spirits,1683


In [22]:
import pandas as pd
import time, os, zipfile

class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self
    def __exit__(self, *args):
        self.end = time.perf_counter()
        self.interval = self.end - self.start

def timeit(fn, n=5):
    times = []
    for _ in range(n):
        with Timer() as t:
            fn()
        times.append(t.interval)
    return sum(times) / len(times)

base = pd.read_csv('https://raw.githubusercontent.com/harmanani/AAI614/main/Week%202/niaaa-report.csv')

sizes = {'1x (original)': 1, '20x': 20, '200x': 200}
results = []

for label, mult in sizes.items():
    df = pd.concat([base]*mult, ignore_index=True)

    df.to_csv('test.csv', index=False)
    df.to_parquet('test.parquet', index=False)
    with zipfile.ZipFile('test.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write('test.csv')

    csv_size = os.path.getsize('test.csv') / 1024
    parquet_size = os.path.getsize('test.parquet') / 1024
    zip_size = os.path.getsize('test.zip') / 1024

    t_csv = timeit(lambda: pd.read_csv('test.csv'))
    t_parquet = timeit(lambda: pd.read_parquet('test.parquet'))
    t_zip = timeit(lambda: pd.read_csv('test.zip', compression='zip'))

    for fmt, size, t in [('CSV', csv_size, t_csv), ('Parquet', parquet_size, t_parquet), ('ZIP', zip_size, t_zip)]:
        results.append({
            'Size': label,
            'Format': fmt,
            'File Size (KB)': round(size, 1),
            'Size vs CSV (%)': round(size / csv_size * 100, 1),
            'Read Time (s)': round(t, 4),
            'Speed vs CSV (%)': round(t / t_csv * 100, 1)
        })

table = pd.DataFrame(results)
table

,Size,Format,File Size (KB),Size vs CSV (%),Read Time (s),Speed vs CSV (%)
0,1x (original),CSV,48.3,100.0,0.0027,100.0
1,1x (original),Parquet,10.8,22.3,0.0037,138.0
2,1x (original),ZIP,11.1,23.0,0.0031,118.2
3,20x,CSV,965.0,100.0,0.0192,100.0
4,20x,Parquet,38.0,3.9,0.0059,30.5
5,20x,ZIP,215.5,22.3,0.0279,145.3
6,200x,CSV,9649.2,100.0,0.1626,100.0
7,200x,Parquet,279.4,2.9,0.0311,19.1
8,200x,ZIP,2152.5,22.3,0.2045,125.7
